In [9]:
from pathlib import Path

rows = []
with Path("data/SMSSpamCollection").open("r", encoding="utf-8") as f:
    for line in f:
        label, text = line.rstrip("\n").split("\t", 1)
        rows.append({"label": label, "text": text})

print(len(rows), rows[0])

5574 {'label': 'ham', 'text': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'}


In [10]:
# !pip -q install fastembed

from fastembed import TextEmbedding

model_name = "BAAI/bge-small-en-v1.5"  # supported + common default
model = TextEmbedding(model_name=model_name)

vec = next(model.embed(["hello world"]))
print(model_name, len(vec))


BAAI/bge-small-en-v1.5 384


In [11]:
# ! export PINECONE_API_KEY="***"

In [12]:
import os
from pinecone import Pinecone, ServerlessSpec

os.environ["PINECONE_API_KEY"] = "***"
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "sms-spam-bge384"
dim = 384

if index_name not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=dim,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

print([i["name"] for i in pc.list_indexes()])


['sms-spam-bge384']


Upsert a small test batch (10 SMS) into Pinecone

In [13]:
from pinecone import Pinecone
from fastembed import TextEmbedding
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index("sms-spam-bge384")

model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

texts = [r["text"] for r in rows[:10]]
labels = [r["label"] for r in rows[:10]]

vecs = list(model.embed(texts))

to_upsert = [
    {"id": str(i), "values": vecs[i], "metadata": {"label": labels[i]}}
    for i in range(10)
]

index.upsert(vectors=to_upsert)

print(index.describe_index_stats())


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '183',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 28 Jan 2026 19:44:09 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '56',
                                    'x-pinecone-request-id': '4472493675648738157',
                                    'x-pinecone-request-latency-ms': '56',
                                    'x-pinecone-response-duration-ms': '57'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}


Query Pinecone and sanity-check nearest neighbors

In [15]:
from fastembed import TextEmbedding

index = pc.Index("sms-spam-bge384")
model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

q_text = rows[11]["text"]
q_vec = next(model.embed([q_text])).tolist()

res = index.query(vector=q_vec, top_k=5, include_metadata=True)
print(q_text)
print(res["matches"])


SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info
[{'id': '5', 'metadata': {'label': 'spam'}, 'score': 0.779377043, 'values': []}, {'id': '8', 'metadata': {'label': 'spam'}, 'score': 0.763393402, 'values': []}, {'id': '2', 'metadata': {'label': 'spam'}, 'score': 0.75509268, 'values': []}, {'id': '7', 'metadata': {'label': 'ham'}, 'score': 0.642994, 'values': []}, {'id': '9', 'metadata': {'label': 'spam'}, 'score': 0.629486144, 'values': []}]


Upsert the FULL dataset (batched)

In [16]:
from fastembed import TextEmbedding

index = pc.Index("sms-spam-bge384")
model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

batch_size = 256

for start in range(0, len(rows), batch_size):
    batch = rows[start:start + batch_size]
    texts = [r["text"] for r in batch]
    labels = [r["label"] for r in batch]

    vecs = [v.tolist() for v in model.embed(texts)]

    to_upsert = [
        {"id": str(start + i), "values": vecs[i], "metadata": {"label": labels[i]}}
        for i in range(len(batch))
    ]
    index.upsert(vectors=to_upsert)

print(index.describe_index_stats())


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '187',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 28 Jan 2026 19:50:11 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-id': '6426776329573648398',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 5574}},
 'storageFullness': 0.0,
 'total_vector_count': 5574,
 'vector_type': 'dense'}


Reserve a few dozen” test queries + sanity-check

In [17]:
from collections import Counter

model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = pc.Index("sms-spam-bge384")

test_ids = [100, 500, 1000, 2000, 3000]

for tid in test_ids:
    q_text = rows[tid]["text"]
    q_label = rows[tid]["label"]
    q_vec = next(model.embed([q_text])).tolist()

    res = index.query(vector=q_vec, top_k=5, include_metadata=True)
    top_labels = [m["metadata"]["label"] for m in res["matches"]]
    print(tid, q_label, Counter(top_labels))


100 ham Counter({'ham': 5})
500 ham Counter({'ham': 5})
1000 ham Counter({'ham': 5})
2000 ham Counter({'ham': 5})
3000 ham Counter({'ham': 5})
